# Chapter 1: Descriptive Statistics — Summarizing Data Honestly

### 1.1 Measures of Central Tendency

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

sales = np.array([98, 105, 112, 98, 125, 130, 145, 112, 98, 160, 135, 118])

print("Mean   :", np.mean(sales))
print("Median :", np.median(sales))
print("Mode   :", stats.mode(sales, keepdims=True).mode[0])

These three numbers all try to answer "what's typical here," but they answer it differently, and picking the wrong one is a genuinely common way analysis misleads decision-makers:

- **Mean** (average) — sums everything and divides by the count. Sensitive to extreme values: one enormous enterprise deal can drag the "average deal size" up in a way that misrepresents what a *typical* deal looks like.
- **Median** — the middle value when sorted. Robust to outliers — this is why median household income, median house price, and median deal size are the standard reporting metric in fields where a few extreme values are common and would otherwise distort a simple average.
- **Mode** — the most frequently occurring value. Most useful for categorical or discrete data ("the most common purchase quantity is 1 unit") and less commonly reported for continuous business metrics.

💡 **The practical rule worth internalizing:** whenever mean and median diverge noticeably, the data is **skewed**, and reporting only the mean is actively misleading. Income, house prices, deal sizes, and customer lifetime value are all classically right-skewed (a long tail of high values pulls the mean above the median) — always check both, and when they disagree, say so explicitly in your write-up rather than quietly picking whichever number looks better.

### 1.2 Measures of Spread

In [ ]:
print("Range              :", np.ptp(sales))                              # max - min
print("Variance           :", np.var(sales))
print("Std Deviation      :", np.std(sales))
print("Coeff of Variation :", (np.std(sales) / np.mean(sales)) * 100, "%")

**Variance** and its square root, **standard deviation**, both measure how spread out the data is around the mean — standard deviation is more commonly reported because it's in the *same units* as the original data (variance is in squared units, which is harder to interpret intuitively: "squared rupees" means nothing to a business audience, while "±₹15,000" does).

The **coefficient of variation** (standard deviation ÷ mean, as a percentage) is the underrated metric here: it lets you compare *consistency* across datasets with completely different scales. "Region A has a standard deviation of ₹50,000, Region B has ₹8,000" sounds like Region B is more consistent — but if Region A's average sales are ₹5,000,000 and Region B's are ₹80,000, Region A is actually *far more consistent relative to its size* (1% CV vs. 10% CV). Always reach for CV, not raw standard deviation, when comparing volatility across groups of different scale.

### 1.3 Percentiles, Quartiles, and the IQR

In [ ]:
print("25th percentile (Q1):", np.percentile(sales, 25))
print("50th percentile (Q2):", np.percentile(sales, 50))     # identical to the median
print("75th percentile (Q3):", np.percentile(sales, 75))
print("90th percentile     :", np.percentile(sales, 90))

q1, q3 = np.percentile(sales, [25, 75])
iqr = q3 - q1
print("IQR:", iqr)

The **N-th percentile** is the value below which N% of the data falls — "the 90th percentile of response time is 400ms" means 90% of requests were faster than that, a standard way SLAs are defined precisely because it's robust to a handful of extreme outliers in a way a simple average response time isn't. The **Interquartile Range** (Q3 − Q1) captures the spread of the "middle 50%" of your data, ignoring the tails entirely — this is exactly what a box plot (Volume 03) visualizes, and it's the basis of the standard outlier detection rule:

In [ ]:
# The classic 1.5×IQR rule for flagging outliers
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = sales[(sales < lower_bound) | (sales > upper_bound)]
print(f"Outlier bounds: [{lower_bound}, {upper_bound}]  |  Outliers found: {outliers}")

This rule (not from any deep statistical theory, but a well-established, widely-used convention introduced by statistician John Tukey) flags anything more than 1.5 IQRs beyond the box edges as worth a second look — not automatically "wrong" or "remove this," but worth investigating before you decide whether it's a data-entry error, a genuinely unusual event, or a legitimate extreme value that belongs in the analysis.

### 1.4 The Normal Distribution

In [ ]:
import matplotlib.pyplot as plt

mu, sigma = 100, 15
data = np.random.normal(mu, sigma, 1000)

plt.figure(figsize=(9, 5))
plt.hist(data, bins=30, density=True, alpha=0.7, color='steelblue', edgecolor='black')

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
plt.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2)     # the theoretical bell curve overlaid on real data
plt.title("Normal Distribution Example (μ=100, σ=15)")
plt.show()

print("Approx % within 1σ:", np.mean((data > mu-sigma) & (data < mu+sigma)) * 100)
print("Approx % within 2σ:", np.mean((data > mu-2*sigma) & (data < mu+2*sigma)) * 100)

The **normal distribution** (bell curve) is defined entirely by two numbers — its mean (μ, "mu") and standard deviation (σ, "sigma") — and it's central to statistics because so many real-world and business phenomena approximate it (measurement errors, aggregated small effects, and, by the Central Limit Theorem covered below, the *average* of almost any large enough sample). The **68-95-99.7 rule** is worth memorizing: roughly 68% of a normal distribution's values fall within 1 standard deviation of the mean, ~95% within 2, and ~99.7% within 3 — a fast mental sanity check any time you're told a data point is "3 standard deviations out" (rare, most likely something worth investigating) versus "1 standard deviation out" (unremarkable).

⚠️ **Not everything is normally distributed, and assuming it is when it isn't leads to bad decisions.** Income, purchase amounts, and time-between-events are all classically right-skewed, not normal — using mean ± standard deviation to describe them (as if they were bell-shaped) can produce nonsensical results, like a "typical range" that includes negative purchase amounts. Always plot the actual distribution (a histogram, per Volume 03) before assuming normality.

### 1.5 Correlation — Relationships Between Variables

In [ ]:
advertising = np.array([10, 15, 20, 25, 30, 35, 40, 45, 50, 55])
sales = np.array([120, 145, 160, 180, 210, 230, 250, 270, 300, 320])

corr = np.corrcoef(advertising, sales)[0, 1]
print(f"Pearson Correlation: {corr:.3f}")

**Pearson correlation** ranges from −1 (perfect inverse relationship) through 0 (no linear relationship) to +1 (perfect direct relationship), and measures specifically the strength of a **linear** relationship between two numeric variables. `np.corrcoef` returns a full correlation *matrix* (correlation of everything with everything, including each variable with itself, which is always 1) — `[0, 1]` picks out the one off-diagonal value you usually actually want.

⚠️ **"Correlation ≠ Causation" is not a cliché to nod at — it's a discipline you need to actively practice**, especially presenting to a business audience primed to jump from "these two things move together" to "one causes the other." Classic counter-examples: ice cream sales and drowning deaths are strongly correlated (both driven by a third factor — hot weather — not by one causing the other); a company's marketing spend and revenue might correlate because a *growing, well-funded* company naturally spends more on both simultaneously, not because the marketing spend is (solely) driving the revenue. Correlation tells you two things move together; only careful experimental design (A/B tests) or causal-inference methods (well beyond this course's scope) can tell you one *causes* the other.

### Business Application: Analyzing a Skewed Real-World Metric

In [ ]:
np.random.seed(42)
purchase_amount = np.random.lognormal(mean=8.5, sigma=0.6, size=500)     # a realistically right-skewed distribution

print("Mean purchase   :", round(np.mean(purchase_amount), 0))
print("Median purchase :", round(np.median(purchase_amount), 0))
print("90th percentile :", round(np.percentile(purchase_amount, 90), 0))
print("% of customers spending > ₹10,000:", round(np.mean(purchase_amount > 10000) * 100, 1), "%")

plt.figure(figsize=(9, 5))
plt.hist(purchase_amount, bins=40, color='steelblue', edgecolor='white')
plt.axvline(np.mean(purchase_amount), color='red', linestyle='--', label='Mean')
plt.axvline(np.median(purchase_amount), color='green', linestyle='--', label='Median')
plt.title("Customer Purchase Amount Distribution")
plt.legend()
plt.show()

Notice the mean sits noticeably above the median here (a hallmark of right-skew, per Section 1.1) — a handful of big-spending customers pull the average up in a way that would overstate what a "typical" customer spends if you reported the mean alone. This exact pattern (log-normal-shaped spend) is extremely common in real e-commerce and B2B transaction data.

### Cheat Sheet — Descriptive Statistics

| Statistic | Function | When to prefer it |
|---|---|---|
| Mean | `np.mean(x)` | Symmetric data, no extreme outliers |
| Median | `np.median(x)` | Skewed data, or outliers present |
| Std Dev | `np.std(x)` | Spread, same units as data |
| Coefficient of Variation | `std/mean * 100` | Comparing spread across different scales |
| Percentile | `np.percentile(x, q)` | SLAs, segmentation cutoffs |
| IQR / outliers | `q3 - q1`; outlier if beyond `Q1 - 1.5*IQR` or `Q3 + 1.5*IQR` | Robust outlier detection |
| Correlation | `np.corrcoef(x, y)[0,1]` | Linear relationship strength, −1 to +1 |

### 🎯 Business Challenge — Descriptive Statistics

In [ ]:
daily_sales = np.array([42, 38, 45, 51, 39, 47, 53, 41, 36, 49,
                         55, 48, 44, 52, 40, 46, 58, 43, 37, 50,
                         54, 47, 45, 49, 56, 42, 39, 51, 48, 53])

mean, median = np.mean(daily_sales), np.median(daily_sales)
mode = stats.mode(daily_sales, keepdims=True).mode[0]
std = np.std(daily_sales)
q1, q3 = np.percentile(daily_sales, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = daily_sales[(daily_sales < lower) | (daily_sales > upper)]
p10, p90 = np.percentile(daily_sales, [10, 90])

print(f"Mean={mean:.1f}  Median={median}  Mode={mode}  Std={std:.2f}  IQR={iqr}")
print(f"Outliers (1.5xIQR rule): {outliers}")
print(f"10th pct={p10}  90th pct={p90}")

plt.hist(daily_sales, bins=10, color='steelblue', edgecolor='black')
plt.title("Daily Sales Distribution (30 Days)")
plt.show()

print("""
Business interpretation: Daily sales cluster tightly around the high-40s
with no statistical outliers detected under the 1.5xIQR rule, suggesting
a stable, predictable demand pattern over this period rather than
sporadic spikes driven by one-off large orders.
""")

---